In [1]:
import numpy as np, pandas as pd, optuna, torch
from sklearn.model_selection import StratifiedKFold
from NN_model import ImprovedNN  # your model
from pathlib import Path
import json, torch

# 0) Load ALREADY-SCALED low-MW data (same feature order as baseline)
df_low_int = pd.read_csv("artifacts/final_train_low&int_MP_scaled_clustered.csv")    # <- already scaled


TARGET_COL = "MP"

exclude = {"SMILES", TARGET_COL, "MW", "MP_category_default", "Structure_Cluster"}
num_cols = df_low_int.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in num_cols if c not in exclude]

X = df_low_int[feature_cols].to_numpy(np.float32)  # <-- already scaled
y = df_low_int[TARGET_COL].to_numpy(np.float32)
y_strat = df_low_int["Structure_Cluster"].astype(str).to_numpy()

# 1) Fix folds once

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
folds = [(tr, va) for tr, va in skf.split(X, y_strat)]


/opt/anaconda3/envs/Surabie_S_clean_v2/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Plot the loss curve 

import matplotlib.pyplot as plt

def plot_training_progress(train_losses, val_losses, early_stop_epoch=None, title="Training and Validation Loss"):
    #train_losses / val_losses: lists of per-epoch average loss values.
    #early_stop_epoch: integer epoch number (1-based) where early stopping triggered (optional).

    epochs = range(1, len(train_losses) + 1) 
    
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, train_losses, label="Training Loss")
    plt.plot(epochs, val_losses,   label="Validation Loss")

    if early_stop_epoch is not None:
        plt.axvline(x=early_stop_epoch, color='r', linestyle='--', label="Early Stop")
    else:
    # draw line at last epoch
        plt.axvline(x=len(train_losses), color='gray', linestyle='--', label="End Epoch")
    
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [3]:
#Source: https://stackoverflow.com/questions/71998978/early-stopping-in-pytorch

# Early Stopping Based on Validation Loss
class EarlyStopper:

    # If the val loss has not been improved (i.e. stayed the same or got worse) for 20 epochs in a row, the training of the model is stopped.
    def __init__(self, patience=30, min_delta=0):
        self.patience = int(patience)
        self.min_delta = float(min_delta)
        self.counter = 0
        self.best_loss = float('inf')
        self.stop = False
        self.stop_epoch = None  # remember which epoch we stopped on (for plotting)

    def early_stop(self, val_loss, epoch=None):

        #For each epoch, checks if the validation loss has improved, we reset the counter.
        # We increase the counter if there is no improvement. Once the counter reaches the patience, we stop and remember the epoch.

        # Improvement means the loss decreased by more than min_delta
        if (self.best_loss - val_loss) > self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            # No meaningful improvement this epoch
            self.counter += 1
            if self.counter >= self.patience:
                self.stop = True
                self.stop_epoch = epoch
        return self.stop
    
import numpy as np
from pathlib import Path
import torch

# sklearn metrics
from sklearn.metrics import r2_score, root_mean_squared_error


def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, y_train, y_val, val_loader, 
                   fold_idx, checkpoint_dir, checkpoint_tracking, is_final=False):
        
        
        # Calculate val predictions
        model.eval()
        all_preds = []
        with torch.no_grad():
            for xb, _ in val_loader:
                preds = model(xb).cpu().numpy()
                all_preds.append(preds)
        preds_val = np.concatenate(all_preds)
        
        # Calculate performance metrics from val predictions
        checkpoint_rmse = root_mean_squared_error(y_val, preds_val)
        checkpoint_r2 = r2_score(y_val, preds_val)
        checkpoint_q2 = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)
        
        # Create checkpoint filename
        if is_final:
            checkpoint_filename = f"checkpoint_epoch_{epoch:03d}_final.pt"
        else:
            checkpoint_filename = f"checkpoint_epoch_{epoch:03d}.pt"
        
        checkpoint_path_full = Path(checkpoint_dir) / checkpoint_filename
        
        # Save the checkpoint
        checkpoint_data = {'epoch': epoch, 'model_state_dict': model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss, 'val_loss': val_loss, 'rmse': checkpoint_rmse, 'r2': checkpoint_r2, 'q2': checkpoint_q2,
            'fold_idx': fold_idx, 'is_final': is_final}
        torch.save(checkpoint_data, checkpoint_path_full)
        
        # Record info for spreadsheet
        checkpoint_info = {'Fold': fold_idx, 'Epoch': epoch, 'Checkpoint_Filename': checkpoint_filename, 'Checkpoint_Path': str(checkpoint_path_full),
            'Train_Loss': round(train_loss, 6), 'Val_Loss': round(val_loss, 6), 'RMSE': round(checkpoint_rmse, 6), 'R2': round(checkpoint_r2, 6), 
            'Q2': round(checkpoint_q2, 6), 'Is_Final': is_final}
        checkpoint_tracking.append(checkpoint_info)
        
        checkpoint_type = "Final" if is_final else "Regular"
        print(f"[Fold {fold_idx}] {checkpoint_type} checkpoint saved at epoch {epoch} - RMSE: {checkpoint_rmse:.4f}")
        return True

import torch
import torch.nn as nn

# since RMSE Loss is not in PyTorch, we define it here using MSELoss

class RMSELoss(nn.Module):
    def __init__(self, eps=1e-8):  

        super().__init__()
        self.mse = nn.MSELoss()
        self.eps = eps      # eps: a small constant to avoid sqrt(0) or division by zero;  to prevent potential numerical instability or "division by zero" like issues if the MSE happens to be exactly zero 

    def forward(self, y_pred, y_true):
        mse = self.mse(y_pred, y_true)
        rmse = torch.sqrt(mse + self.eps)
        return rmse

        
# ==== Standard libraries ====
import copy
from pathlib import Path

# ==== Numerical ====
import numpy as np

# ==== PyTorch ====
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ==== sklearn metrics ====
from sklearn.metrics import r2_score, root_mean_squared_error

# ==== Your custom modules ====
from NN_model import ImprovedNN   # make sure NN_model/ folder is in sys.path
import copy 

def evaluate_fold(trial, fold_idx, X_train_scaled, y_train, X_val_scaled, y_val, hidden_layers, learning_rate, batch_size, dropout_rate, weight_decay, max_epochs = 10**9, patience = 30, min_delta = 0, X_test_scaled=None, y_test=None, save_checkpoints=True, checkpoint_dir="checkpoints", save_every_n_epochs=15):

    # Set device to CPU
    device = torch.device("cpu")
    print(f"Fold {fold_idx}: Training on cpu")

    #Setup checkpoint directory and tracking list
    checkpoint_tracking = []  # Empty list to track performance metrics for model checkpointing
    
    # If saving checkpoints is true, we are creating the path checkpoints/fold_{fold_idx}
    if save_checkpoints:
        checkpoint_path = Path(checkpoint_dir)
        checkpoint_path.mkdir(parents=True, exist_ok=True)
        fold_checkpoint_dir = checkpoint_path / f"fold_{fold_idx}"
        fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
        print(f"Checkpoints will be saved to: {fold_checkpoint_dir}")

    # Convert data to tensors and move to device
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32).to(device)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32).to(device)      # reshape the targets to column vectors to match the model’s predictions and prevent PyTorch from doing sneaky broadcasting
    X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32).to(device)
    y_val_tensor   = torch.tensor(y_val,   dtype=torch.float32).to(device)


    # Load the training df
    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    
    #Load the val df 
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)   
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    #model, optimizer  scheduler, loss (set up training components)
    model     = ImprovedNN(input_size = X_train_scaled.shape[1], hidden_layers=hidden_layers, dropout_rate = dropout_rate).to(device) #A new model is created for each trial run with Optuna, the hyperparameters in each trial is chosen by Optuna, new instance of the model is created, and input size is determined by features in scaled training data, drop out rate is suggested by Optuna
    criterion = RMSELoss() # changed from HuberLoss to RMSELoss 
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay) #Optimizer adjusts the model's internal weights and biases, AdamW is an optimizer, model.parameters() tells optimizer what to optimize, lr = learning_rate uses suggested learning rate by Optuna, same for weight_decay                     
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10) #Automatically adjusts learning rate during training, mode = "min" monitors metric to minimize, factor = 0.5 if monitored metric doesn't improve for a certain amount of epochs reduce lr by 1/2, patience is number of epochs to wait before adjustment by factor 
                                               
    # Set up values for early stopping
    early_stopper = EarlyStopper(patience=patience, min_delta=min_delta)

    best_val_loss = float('inf')
    best_state = copy.deepcopy(model.state_dict())

    train_losses, val_losses = [], []
    stop_epoch = None

        #-- Model Training ---
    for epoch in range(1, max_epochs + 1): ##for loop represemts the training process for a single model for the current trial, runs for 300 epochs, each epoch indicates that the model has run once, so 12 epoches means the model has been run 12 times 
        model.train()
        train_loss = 0.0
        for xb, yb in train_loader: ##Put model into training mode (dropout turn on), loops over each batch (xb = input, yb = target)
            optimizer.zero_grad() #Clear out any old gradients (a gradient is a piece of information about how much much to change the weights)
            preds = model(xb) #make predictions
            loss  = criterion(preds, yb) #Calculate loss function
            loss.backward() #Back propogate
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5) #Prevents exploding gradients which causes the model to become unstable, limits how big the adjustments to weights can be 
            optimizer.step() #Uses calculated gradients to actually update model's weights and biases trying to reduce loss 
            train_loss += loss.item()
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # --- To validate the model ----
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in val_loader:
                preds = model(xb)
                loss  = criterion(preds, yb)
                val_loss += loss.item()
        val_loss /= len(val_loader)
        val_losses.append(val_loss)   
       
        scheduler.step(val_loss)
        
        # Update best model if validation loss improves
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
        
        # Saves checkpoints every n epochs (and at epoch 1)
        if save_checkpoints and (epoch % save_every_n_epochs == 0 or epoch == 1):
            # Calculate metrics for this checkpoint
            save_checkpoint(model, optimizer, epoch, train_loss, val_loss, y_train, y_val, val_loader, fold_idx, fold_checkpoint_dir, checkpoint_tracking, is_final=False)

        # Check for early stopping
        should_stop = early_stopper.early_stop(val_loss, epoch=epoch)
        if should_stop:
            stop_epoch = early_stopper.stop_epoch
            print(f"[Fold {fold_idx}] Early stopping  at epoch {stop_epoch} (best Val Loss: {best_val_loss:.4f})")

            # Save final checkpoint on early stop (guarantee last snapshot)
            if save_checkpoints and epoch % save_every_n_epochs != 0 and epoch != 1:
                save_checkpoint(model, optimizer, epoch, train_loss, val_loss, y_train, y_val, 
                              val_loader, fold_idx, fold_checkpoint_dir, checkpoint_tracking, is_final=True)
            
            break

        # Log progress every 50 epochs or first epoch
        if epoch % 50 == 0 or epoch == 1:
            print(f"[Fold {fold_idx}] Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | ES {early_stopper.counter}/{patience}")
    
    # Load best model state (from epoch with lowest val loss)
    model.load_state_dict(best_state)
    model.eval()  

    # Save the checkpoint tracking spreadsheet for this fold
    if save_checkpoints and checkpoint_tracking:
        df_checkpoints = pd.DataFrame(checkpoint_tracking)
        spreadsheet_file = fold_checkpoint_dir / f"fold_{fold_idx}_checkpoints_low&int_test.csv"
        df_checkpoints.to_csv(spreadsheet_file, index=False)
        print(f"[Fold {fold_idx}] Checkpoint spreadsheet saved: {spreadsheet_file}")
        print(f"[Fold {fold_idx}] Total checkpoints saved: {len(checkpoint_tracking)}")
  

    # Final metrics calculation (using the best model)
    all_preds = []
    with torch.no_grad():
        for xb, _ in val_loader:
            preds = model(xb).cpu().numpy()
            all_preds.append(preds)
    preds_val = np.concatenate(all_preds)
    
    rmse = root_mean_squared_error(y_val, preds_val)
    r2 = r2_score(y_val, preds_val)
    q2 = 1 - np.sum((y_val - preds_val)**2) / np.sum((y_val - y_train.mean())**2)
 
    return rmse, r2, q2, model, train_losses, val_losses, stop_epoch


import time
import numpy as np
import torch

# Step 3: Hyperparameter optimization
trial_times = []

def objective(trial):
    # Suggest hyperparameters
    dropout_rate  = trial.suggest_float("dropout_rate",  0.2, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    weight_decay  = trial.suggest_float("weight_decay",  1e-6, 1e-2, log=True)
    batch_size    = trial.suggest_categorical("batch_size", [16, 32, 64])

    # First hidden layer max 256
    h1 = trial.suggest_categorical("h1", [64, 96, 128, 160, 192, 224, 256])
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    hidden_layers = [h1, h2, h3]

    start = time.perf_counter()

    rmses = []

    # Use folds you defined earlier
    for fold_idx, (tr_idx, val_idx) in enumerate(folds):
        X_train_scaled = X[tr_idx]
        y_train        = y[tr_idx]
        X_val_scaled   = X[val_idx]
        y_val          = y[val_idx]
        
        trial_checkpoint_root = Path("checkpoints_low&int_MP") / f"trial_{trial.number:03d}"

        rmse, _, _, _, _, _, _ = evaluate_fold(
            trial=trial,
            fold_idx=fold_idx,
            X_train_scaled=X_train_scaled,
            y_train=y_train,
            X_val_scaled=X_val_scaled,
            y_val=y_val,
            learning_rate=learning_rate,
            batch_size=batch_size,
            hidden_layers=hidden_layers,
            dropout_rate=dropout_rate,
            weight_decay=weight_decay,
            save_checkpoints=True,
            checkpoint_dir=trial_checkpoint_root
        )
        rmses.append(rmse)

    elapsed = (time.perf_counter() - start) / 60.0
    trial_times.append(elapsed)
    print(f"Trial {trial.number} finished in {elapsed:.2f} minutes")

    avg_rmse = float(np.mean(rmses))
    print(f"Trial {trial.number}: Average RMSE = {avg_rmse:.4f}")
    return avg_rmse

In [4]:
# Plot the loss curve 

import matplotlib.pyplot as plt

def plot_training_progress(train_losses, val_losses, early_stop_epoch=None, title="Training and Validation Loss"):
    #train_losses / val_losses: lists of per-epoch average loss values.
    #early_stop_epoch: integer epoch number (1-based) where early stopping triggered (optional).

    epochs = range(1, len(train_losses) + 1) 
    
    plt.figure(figsize=(8, 4))
    plt.plot(epochs, train_losses, label="Training Loss")
    plt.plot(epochs, val_losses,   label="Validation Loss")

    if early_stop_epoch is not None:
        plt.axvline(x=early_stop_epoch, color='r', linestyle='--', label="Early Stop")
    else:
    # draw line at last epoch
        plt.axvline(x=len(train_losses), color='gray', linestyle='--', label="End Epoch")
    
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [5]:
import time 
import torch
import optuna
from optuna.importance import get_param_importances
import optuna.visualization as vis 
import copy

device = torch.device("cpu")

def set_optuna_study(n_trials): 
    start_time = time.perf_counter()
    print("Setting up Optuna study...")
    
    # 1) Set up the Optuna study
    study = optuna.create_study(direction='minimize') #minimize return loss
    study.optimize(objective, n_trials=n_trials)  #CHANGE TO 100 AFTER TESTING
    
    # 2) Identify the best hyperparameters
    best_params = study.best_params #best_params holds the dropout, learning rate, and weight decay that gave the lowest best_val_loss
    print("Best hyperparameters:", best_params)
    
    end_time = time.perf_counter()
    elapsed_time = (end_time - start_time) / 60.0
    print(f"Optuna study completed in {elapsed_time:.2f} minutes")
    
    return best_params, study

best_params, study = set_optuna_study(n_trials=20) 

[I 2025-12-03 09:21:45,994] A new study created in memory with name: no-name-9b469fc1-e310-47f5-8d4a-c4685d94c563


Setting up Optuna study...
Fold 0: Training on cpu
Checkpoints will be saved to: checkpoints_low&int_MP/trial_000/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 123.7162
[Fold 0] Epoch    1 | Train Loss: 126.0395 | Val Loss: 123.2818 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 99.5322
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 62.2680
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 38.1875
[Fold 0] Epoch   50 | Train Loss: 46.0818 | Val Loss: 36.6017 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 35.9206
[Fold 0] Regular checkpoint saved at epoch 75 - RMSE: 35.1905
[Fold 0] Regular checkpoint saved at epoch 90 - RMSE: 35.5886
[Fold 0] Epoch  100 | Train Loss: 42.9725 | Val Loss: 34.1854 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 105 - RMSE: 35.3035
[Fold 0] Regular checkpoint saved at epoch 120 - RMSE: 35.3473
[Fold 0] Early stopping  at epoch 130 (best Val Loss: 34.1854)
[Fold 0] Final checkpoint saved at epo

[I 2025-12-03 09:35:31,008] Trial 0 finished with value: 35.644979095458986 and parameters: {'dropout_rate': 0.29744443769361445, 'learning_rate': 0.0001198924317113314, 'weight_decay': 1.671051255176672e-06, 'batch_size': 32, 'h1': 96}. Best is trial 0 with value: 35.644979095458986.


[Fold 9] Early stopping  at epoch 208 (best Val Loss: 34.2922)
[Fold 9] Final checkpoint saved at epoch 208 - RMSE: 35.2280
[Fold 9] Checkpoint spreadsheet saved: checkpoints_low&int_MP/trial_000/fold_9/fold_9_checkpoints_low&int_test.csv
[Fold 9] Total checkpoints saved: 15
Trial 0 finished in 13.75 minutes
Trial 0: Average RMSE = 35.6450
Fold 0: Training on cpu
Checkpoints will be saved to: checkpoints_low&int_MP/trial_001/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 124.4515
[Fold 0] Epoch    1 | Train Loss: 125.9260 | Val Loss: 123.4950 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 121.2123
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 113.4106
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 103.0478
[Fold 0] Epoch   50 | Train Loss: 102.2605 | Val Loss: 101.5252 | ES 2/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 93.8745
[Fold 0] Regular checkpoint saved at epoch 75 - RMSE: 81.8454
[Fold 0] Regular checkpoint saved at epoc

[I 2025-12-03 10:08:23,747] Trial 1 finished with value: 40.12806510925293 and parameters: {'dropout_rate': 0.35972204385625567, 'learning_rate': 1.7570089529529226e-05, 'weight_decay': 0.00014747734439321346, 'batch_size': 16, 'h1': 96}. Best is trial 0 with value: 35.644979095458986.


...

[Fold 9] Early stopping  at epoch 111 (best Val Loss: 33.9490)
[Fold 9] Final checkpoint saved at epoch 111 - RMSE: 35.0640
[Fold 9] Checkpoint spreadsheet saved: checkpoints_low&int_MP/trial_017/fold_9/fold_9_checkpoints_low&int_test.csv
[Fold 9] Total checkpoints saved: 9
Trial 17 finished in 26.29 minutes
Trial 17: Average RMSE = 33.7206
Fold 0: Training on cpu
Checkpoints will be saved to: checkpoints_low&int_MP/trial_018/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 113.1825
[Fold 0] Epoch    1 | Train Loss: 120.7626 | Val Loss: 113.0642 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 34.9747
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 33.7576
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 33.6544
[Fold 0] Epoch   50 | Train Loss: 33.1603 | Val Loss: 33.9297 | ES 3/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 33.5904
[Fold 0] Regular checkpoint saved at epoch 75 - RMSE: 33.7185
[Fold 0] Early stopping  at epoch 77 (best Va

[I 2025-12-03 17:05:15,058] Trial 18 finished with value: 33.77115364074707 and parameters: {'dropout_rate': 0.27512919207832726, 'learning_rate': 0.000880277073359873, 'weight_decay': 1.2732766111347321e-06, 'batch_size': 64, 'h1': 256}. Best is trial 17 with value: 33.7205867767334.


[Fold 9] Regular checkpoint saved at epoch 90 - RMSE: 34.1780
[Fold 9] Early stopping  at epoch 90 (best Val Loss: 33.4531)
[Fold 9] Checkpoint spreadsheet saved: checkpoints_low&int_MP/trial_018/fold_9/fold_9_checkpoints_low&int_test.csv
[Fold 9] Total checkpoints saved: 7
Trial 18 finished in 18.57 minutes
Trial 18: Average RMSE = 33.7712
Fold 0: Training on cpu
Checkpoints will be saved to: checkpoints_low&int_MP/trial_019/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 97.0304
[Fold 0] Epoch    1 | Train Loss: 112.7362 | Val Loss: 96.6642 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 34.5733
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 33.7052
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 33.9807
[Fold 0] Epoch   50 | Train Loss: 34.8307 | Val Loss: 33.2675 | ES 18/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 33.4845
[Fold 0] Regular checkpoint saved at epoch 75 - RMSE: 33.3383
[Fold 0] Regular checkpoint saved at epoch 90 

[I 2025-12-03 17:30:48,291] Trial 19 finished with value: 33.78842391967773 and parameters: {'dropout_rate': 0.28291025451852864, 'learning_rate': 0.0009257070607626474, 'weight_decay': 1.0545431721049914e-06, 'batch_size': 32, 'h1': 256}. Best is trial 17 with value: 33.7205867767334.


[Fold 9] Early stopping  at epoch 158 (best Val Loss: 33.3528)
[Fold 9] Final checkpoint saved at epoch 158 - RMSE: 34.1900
[Fold 9] Checkpoint spreadsheet saved: checkpoints_low&int_MP/trial_019/fold_9/fold_9_checkpoints_low&int_test.csv
[Fold 9] Total checkpoints saved: 12
Trial 19 finished in 25.55 minutes
Trial 19: Average RMSE = 33.7884
Best hyperparameters: {'dropout_rate': 0.2522260224901551, 'learning_rate': 0.0009683138191375643, 'weight_decay': 0.00012041535170169376, 'batch_size': 32, 'h1': 256}
Optuna study completed in 489.04 minutes


In [6]:
# Save the best parameters

print("Best Trial Number:", study.best_trial.number)
print("  RMSE:", study.best_value)
print("  Params:", study.best_params)

Best Trial Number: 17
  RMSE: 33.7205867767334
  Params: {'dropout_rate': 0.2522260224901551, 'learning_rate': 0.0009683138191375643, 'weight_decay': 0.00012041535170169376, 'batch_size': 32, 'h1': 256}


In [7]:
from pathlib import Path
import torch
import pandas as pd

# BASE and artifacts_dir should already be defined (same script as before)
BASE = Path.cwd()  # transfer_learning
artifacts_dir = BASE / "artifacts"

# ---------- Directories for final best models + checkpoints ----------
# (Rename to MP if you want, but keeping your original name for now)
best_models_dir = artifacts_dir / "low&int_MP_best_models"
best_models_dir.mkdir(parents=True, exist_ok=True)

final_ckpt_dir = BASE / "checkpoints_low&int_MP_best"
final_ckpt_dir.mkdir(parents=True, exist_ok=True)

# Make sure best_params exists (from your Optuna study: best_params, study = set_optuna_study(...))
print("Best hyperparameters from Optuna:", best_params)

# Helper to derive hidden layers from best_params (same logic as in objective)
def build_hidden_layers_from_best(best_params):
    h1 = best_params["h1"]
    h2 = max(h1 // 2, 4)
    h3 = max(h2 // 2, 2)
    return [h1, h2, h3]

hidden_layers = build_hidden_layers_from_best(best_params)
dropout_rate  = best_params["dropout_rate"]
learning_rate = best_params["learning_rate"]
weight_decay  = best_params["weight_decay"]
batch_size    = best_params["batch_size"]

print("Using hidden_layers:", hidden_layers)
print("dropout:", dropout_rate, "| lr:", learning_rate, "| wd:", weight_decay, "| batch_size:", batch_size)

# To keep track of metrics across folds
final_fold_metrics = []

# ---------- Final training loop for all folds (using `folds`, X, y) ----------
# Assumes you already defined:
#   X, y, folds = [(tr_idx, val_idx), ...] earlier (same as in `objective`)
for fold_idx, (tr_idx, val_idx) in enumerate(folds):
    print(f"\n==================== Final training for fold {fold_idx} ====================")

    # Slice the global X, y using the same folds as during Optuna
    X_train_scaled = X[tr_idx]
    y_train        = y[tr_idx]
    X_val_scaled   = X[val_idx]
    y_val          = y[val_idx]

    # Train this fold with the best hyperparameters
    rmse, r2, q2, model, train_losses, val_losses, stop_epoch = evaluate_fold(
        trial=None,                    # not needed for final run
        fold_idx=fold_idx,
        X_train_scaled=X_train_scaled,
        y_train=y_train,
        X_val_scaled=X_val_scaled,
        y_val=y_val,
        hidden_layers=hidden_layers,
        learning_rate=learning_rate,
        batch_size=batch_size,
        dropout_rate=dropout_rate,
        weight_decay=weight_decay,
        max_epochs=10**9,              # will stop via early-stopping anyway
        patience=30,
        min_delta=0.0,
        save_checkpoints=True,         # save checkpoints for this BEST config
        checkpoint_dir=final_ckpt_dir, # root; evaluate_fold will create fold_{k}/ inside
        save_every_n_epochs=15
    )

    # ---------- Save the final (best) model for this fold ----------
    model_path = best_models_dir / f"low&int_MP_best_fold_{fold_idx}.pt"
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "hidden_layers": hidden_layers,
            "dropout_rate": dropout_rate,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "batch_size": batch_size,
            "fold_idx": fold_idx,
            "rmse": rmse,
            "r2": r2,
            "q2": q2,
        },
        model_path,
    )
    print(f"[Fold {fold_idx}] Saved best model to: {model_path}")

    # store metrics
    final_fold_metrics.append(
        {
            "Fold": fold_idx,
            "RMSE": rmse,
            "R2": r2,
            "Q2": q2,
            "Stop_Epoch": stop_epoch,
            "Model_Path": str(model_path),
        }
    )

# ---------- Save a summary CSV of all folds ----------
metrics_df = pd.DataFrame(final_fold_metrics)
metrics_path = best_models_dir / "low&int_MP_best_models_summary.csv"
metrics_df.to_csv(metrics_path, index=False)
print("\n✅ Saved summary of best models across folds to:", metrics_path)
print(metrics_df)


Best hyperparameters from Optuna: {'dropout_rate': 0.2522260224901551, 'learning_rate': 0.0009683138191375643, 'weight_decay': 0.00012041535170169376, 'batch_size': 32, 'h1': 256}
Using hidden_layers: [256, 128, 64]
dropout: 0.2522260224901551 | lr: 0.0009683138191375643 | wd: 0.00012041535170169376 | batch_size: 32

==================== Final training for fold 0 ====================
Fold 0: Training on cpu
Checkpoints will be saved to: /Users/sdl5_mp/Documents/GitHub/SDL5_MP/transfer_learning/checkpoints_low&int_MP_best/fold_0
[Fold 0] Regular checkpoint saved at epoch 1 - RMSE: 97.0200
[Fold 0] Epoch    1 | Train Loss: 113.9006 | Val Loss: 96.6661 | ES 0/30
[Fold 0] Regular checkpoint saved at epoch 15 - RMSE: 34.5320
[Fold 0] Regular checkpoint saved at epoch 30 - RMSE: 35.0739
[Fold 0] Regular checkpoint saved at epoch 45 - RMSE: 34.4619
[Fold 0] Epoch   50 | Train Loss: 34.6815 | Val Loss: 33.9780 | ES 16/30
[Fold 0] Regular checkpoint saved at epoch 60 - RMSE: 34.1370
[Fold 0] Re

In [ ]:
# Plotting optuna trial with RMSE

import matplotlib.pyplot as plt
import pandas as pd

# Collect data from your study
records = []
for t in study.trials:
    if t.value is not None:  # skip failed/pruned trials
        records.append({"trial": t.number, "rmse": t.value})

df = pd.DataFrame(records)

plt.figure(figsize=(8,5))
plt.plot(df["trial"], df["rmse"], marker="o")
plt.title("Optuna Trials vs Mean RMSE (10-Fold Average)")
plt.xlabel("Trial Number")
plt.ylabel("Average RMSE")
plt.grid(True)
plt.show()

In [ ]:
# For Each trial, save the model for each fold?
# when saving checkpoints, is it saving the whole model?
# differenc between epochs saving and the model saving



In [ ]:
# Load the best model (best trial, best fold)


# Grab the test data and run it